# Pipeline ETL - Etapas A até D
## Consolidação de Bases Anonimizadas SUSEP

Este notebook implementa as etapas A até D do plano de consolidação:
- **Etapa A**: Catalogação (geração de manifestos)
- **Etapa B**: Download e Integridade (cálculo de checksums)
- **Etapa C**: Extração (descompactar ZIPs)
- **Etapa D**: Padronização de Esquema (standardização de dados)

Baseado em: arquivos já baixados em `data/`

## Seção 1: Importar Bibliotecas e Configurar Setup

In [1]:
import os
import csv
import hashlib
import zipfile
import pandas as pd
from pathlib import Path
from datetime import datetime
import logging
from typing import Dict, List, Tuple
import shutil

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Diretórios base
DATA_ROOT = Path("data/gov_br/susep/central-de-conteudos/dados-estatisticos/bases-anonimizadas")
UNIFIED_ROOT = Path("data/unified")

# Criar diretórios se não existirem
(UNIFIED_ROOT / "manifestos").mkdir(parents=True, exist_ok=True)

# Mapeamento de ramos e grupos de arquivos
RAMOS = {
    "bases_auto": {"ramo": "auto", "grupos": ["R_AUTO", "S_AUTO"]},
    "bases_comp": {"ramo": "comp", "grupos": ["R_COMP", "S_COMP"]},
    "bases_rural": {"ramo": "rural", "grupos": ["R_RURAL", "S_RURAL"]},
}

print("✓ Bibliotecas importadas e diretórios configurados")

✓ Bibliotecas importadas e diretórios configurados


## Seção 2: Etapa A - Catalogação (Geração de Manifestos)

In [2]:
def calcular_sha256(filepath: Path) -> str:
    """Calcula SHA256 de um arquivo."""
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()


def etapa_a_catalogacao():
    """Etapa A: Catalogação - gera manifestos de links e checksums."""
    print("\n" + "="*70)
    print("ETAPA A - CATALOGAÇÃO")
    print("="*70)
    
    manifestos = []
    
    for base_folder, base_info in RAMOS.items():
        base_path = DATA_ROOT / base_folder
        ramo = base_info["ramo"]
        
        print(f"\nProcessando ramo: {ramo} ({base_folder})")
        
        for grupo in base_info["grupos"]:
            grupo_path = base_path / "raw" / grupo
            
            if not grupo_path.exists():
                print(f"  ⚠ Grupo {grupo} não encontrado em {grupo_path}")
                continue
            
            # Iterar sobre períodos/subpastas
            for periodo_folder in sorted(grupo_path.iterdir()):
                if not periodo_folder.is_dir():
                    continue
                
                periodo = periodo_folder.name
                
                # Procurar ZIPs nesta subpasta
                for zip_file in sorted(periodo_folder.glob("*.zip")):
                    filepath = zip_file
                    tamanho = filepath.stat().st_size
                    sha256 = calcular_sha256(filepath)
                    
                    manifestos.append({
                        "ramo": ramo,
                        "grupo_arquivo": grupo,
                        "periodo_bruto": periodo,
                        "ano": int(periodo[:4]),
                        "semestre": periodo[4] if len(periodo) > 4 else None,
                        "url_origem": "",
                        "nome_arquivo_zip": zip_file.name,
                        "sha256_publicado": "",
                        "sha256_calculado": sha256,
                        "data_download_utc": datetime.utcnow().isoformat(),
                        "tamanho_bytes": tamanho,
                        "status_download": "concluido",
                        "status_checksum": "calculado",
                    })
                    
                    tamanho_mb = tamanho / 1e6
                    print(f"  ✓ [{grupo}/{periodo}] {zip_file.name:30} {tamanho_mb:>8.1f} MB")
    
    # Salvar manifesto consolidado
    manifest_path = UNIFIED_ROOT / "manifestos"
    manifest_path.mkdir(parents=True, exist_ok=True)
    
    manifest_file = manifest_path / "links_checksums_manifest.csv"
    
    if manifestos:
        df_manifest = pd.DataFrame(manifestos)
        df_manifest.to_csv(manifest_file, index=False, quoting=csv.QUOTE_ALL)
        print(f"\n✓ Manifesto salvo: {manifest_file}")
        print(f"  Total de arquivos catalogados: {len(manifestos)}")
        return df_manifest
    else:
        print("⚠ Nenhum arquivo ZIP encontrado!")
        return None


# Executar Etapa A
df_manifest = etapa_a_catalogacao()


ETAPA A - CATALOGAÇÃO

Processando ramo: auto (bases_auto)


/tmp/ipykernel_733785/4250453404.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "data_download_utc": datetime.utcnow().isoformat(),


  ✓ [R_AUTO/2020A] R_AUTO_2020A.zip                 1786.8 MB
  ✓ [R_AUTO/2020B] R_AUTO_2020B.zip                 1656.8 MB
  ✓ [S_AUTO/2020A] S_AUTO_2020A.zip                   79.4 MB
  ✓ [S_AUTO/2020B] S_AUTO_2020B.zip                   63.3 MB

Processando ramo: comp (bases_comp)
  ✓ [R_COMP/2022] R_COMP_2022.zip                  1743.1 MB
  ✓ [S_COMP/2022] S_COMP_2022.zip                    11.1 MB

Processando ramo: rural (bases_rural)
  ✓ [R_RURAL/2021] R_RURAL_2021.zip                  254.6 MB
  ✓ [S_RURAL/2021] S_RURAL_2021.zip                    3.8 MB

✓ Manifesto salvo: data/unified/manifestos/links_checksums_manifest.csv
  Total de arquivos catalogados: 8


## Seção 3: Etapa B - Download e Integridade (Verificação de Checksums)

In [3]:
def etapa_b_download_integridade():
    """Etapa B: Download e Integridade - valida checksums."""
    print("\n" + "="*70)
    print("ETAPA B - DOWNLOAD E INTEGRIDADE")
    print("="*70)
    
    manifest_file = UNIFIED_ROOT / "manifestos" / "links_checksums_manifest.csv"
    
    if not manifest_file.exists():
        print("⚠ Arquivo de manifesto não encontrado!")
        return None
    
    df = pd.read_csv(manifest_file)
    
    print(f"\n✓ Validação de checksums:")
    print(f"  Total de arquivos a validar: {len(df)}")
    
    # Revalidar todos os checksums
    for idx, row in df.iterrows():
        ramo = row['ramo']
        grupo = row['grupo_arquivo']
        periodo = row['periodo_bruto']
        nome_zip = row['nome_arquivo_zip']
        
        base_folder = next((k for k, v in RAMOS.items() if v['ramo'] == ramo), None)
        if not base_folder:
            continue
        
        base_path = DATA_ROOT / base_folder
        zip_path = base_path / "raw" / grupo / periodo / nome_zip
        
        if zip_path.exists():
            sha256_calc = calcular_sha256(zip_path)
            df.at[idx, 'sha256_calculado'] = sha256_calc
            print(f"  ✓ {nome_zip:40} {sha256_calc[:16]}...")
        else:
            print(f"  ✗ Arquivo não encontrado: {zip_path}")
    
    # Salvar manifesto atualizado
    df.to_csv(manifest_file, index=False, quoting=csv.QUOTE_ALL)
    print(f"\n✓ Manifesto atualizado: {manifest_file}")
    
    return df


# Executar Etapa B
df_manifest = etapa_b_download_integridade()


ETAPA B - DOWNLOAD E INTEGRIDADE

✓ Validação de checksums:
  Total de arquivos a validar: 8
  ✓ R_AUTO_2020A.zip                         fa08183405b38672...
  ✓ R_AUTO_2020B.zip                         b68175110747c51f...
  ✓ S_AUTO_2020A.zip                         b6a999e87f4198d1...
  ✓ S_AUTO_2020B.zip                         6c4d6df6e3229d59...
  ✓ R_COMP_2022.zip                          90aa6b3ae143d2fc...
  ✓ S_COMP_2022.zip                          fd6fe781a1bd7b9c...
  ✓ R_RURAL_2021.zip                         03971db74e92a7bf...
  ✓ S_RURAL_2021.zip                         9648761e70a7381b...

✓ Manifesto atualizado: data/unified/manifestos/links_checksums_manifest.csv


## Seção 4: Etapa C - Extração (Descompactação de ZIPs)

In [4]:
def etapa_c_extracao():
    """Etapa C: Extração - descompacta ZIPs."""
    print("\n" + "="*70)
    print("ETAPA C - EXTRAÇÃO")
    print("="*70)
    
    total_extraido = 0
    sucessos = 0
    erros = 0
    
    for base_folder, base_info in RAMOS.items():
        base_path = DATA_ROOT / base_folder
        ramo = base_info["ramo"]
        
        print(f"\nExtraindo ramo: {ramo}")
        
        for grupo in base_info["grupos"]:
            grupo_path = base_path / "raw" / grupo
            
            if not grupo_path.exists():
                continue
            
            for periodo_folder in sorted(grupo_path.iterdir()):
                if not periodo_folder.is_dir():
                    continue
                
                periodo = periodo_folder.name
                
                for zip_file in sorted(periodo_folder.glob("*.zip")):
                    # Criar pasta de extração
                    extract_path = base_path / "extracted" / grupo / periodo
                    extract_path.mkdir(parents=True, exist_ok=True)
                    
                    try:
                        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                            zip_ref.extractall(extract_path)
                        
                        # Contar arquivos extraídos
                        num_files = len([f for f in extract_path.rglob('*') if f.is_file()])
                        total_extraido += num_files
                        sucessos += 1
                        
                        print(f"  ✓ [{grupo}/{periodo}] {zip_file.name:40} ({num_files} arquivos)")
                    except Exception as e:
                        erros += 1
                        print(f"  ✗ [{grupo}/{periodo}] Erro: {str(e)[:50]}")
    
    print(f"\n✓ Extração completa!")
    print(f"  Sucessos: {sucessos} | Erros: {erros} | Total de arquivos: {total_extraido}")


# Executar Etapa C
etapa_c_extracao()


ETAPA C - EXTRAÇÃO

Extraindo ramo: auto
  ✓ [R_AUTO/2020A] R_AUTO_2020A.zip                         (1 arquivos)
  ✓ [R_AUTO/2020B] R_AUTO_2020B.zip                         (1 arquivos)
  ✓ [S_AUTO/2020A] S_AUTO_2020A.zip                         (1 arquivos)
  ✓ [S_AUTO/2020B] S_AUTO_2020B.zip                         (1 arquivos)

Extraindo ramo: comp
  ✓ [R_COMP/2022] R_COMP_2022.zip                          (1 arquivos)
  ✓ [S_COMP/2022] S_COMP_2022.zip                          (1 arquivos)

Extraindo ramo: rural
  ✓ [R_RURAL/2021] R_RURAL_2021.zip                         (1 arquivos)
  ✓ [S_RURAL/2021] S_RURAL_2021.zip                         (1 arquivos)

✓ Extração completa!
  Sucessos: 8 | Erros: 0 | Total de arquivos: 8


## Seção 5: Etapa D - Padronização de Esquema

In [8]:
import pyarrow as pa
import pyarrow.csv as pa_csv
import pyarrow.parquet as pq


def etapa_d_padronizacao():
    """Etapa D: Padronização de Esquema.

    Usa PyArrow CSV (streaming em batches) para lidar com CSVs de até 12 GB.
    Os CSVs da SUSEP usam delimitador ';' e encoding UTF-8.
    Todas as colunas são lidas como string para evitar erros de inferência de tipo
    (valores como '.', 'CE', 'RJ' em colunas mistas).
    """
    print("\n" + "="*70)
    print("ETAPA D - PADRONIZAÇÃO DE ESQUEMA")
    print("="*70)

    manifest_file = UNIFIED_ROOT / "manifestos" / "links_checksums_manifest.csv"
    df_manifest = pd.read_csv(manifest_file)

    sucessos = 0
    erros = 0

    # Opções de leitura CSV PyArrow: delimitador ';'
    parse_options = pa_csv.ParseOptions(delimiter=';')
    read_options = pa_csv.ReadOptions(encoding='utf-8')
    # Forçar todas as colunas como string para evitar erros de conversão
    convert_options = pa_csv.ConvertOptions(
        strings_can_be_null=True,
        column_types={},  # será preenchido após ler o header
    )

    for _, row in df_manifest.iterrows():
        ramo = row['ramo']
        grupo = row['grupo_arquivo']
        periodo = row['periodo_bruto']

        base_folder = next((k for k, v in RAMOS.items() if v['ramo'] == ramo), None)
        if not base_folder:
            continue

        base_path = DATA_ROOT / base_folder
        extract_path = base_path / "extracted" / grupo / periodo

        if not extract_path.exists():
            print(f"  ⚠ Pasta não encontrada: {extract_path}")
            continue

        csv_files = list(extract_path.glob("*.csv"))
        if not csv_files:
            print(f"  ⚠ Nenhum CSV em {extract_path}")
            continue

        for csv_file in csv_files:
            try:
                # Destino do Parquet padronizado
                standardized_path = base_path / "standardized"
                standardized_path.mkdir(parents=True, exist_ok=True)
                parquet_file = standardized_path / f"{csv_file.stem}_{periodo}_standardized.parquet"

                # Colunas de linhagem
                ano_val = int(periodo[:4])
                semestre_val = periodo[4] if len(periodo) > 4 else None

                # 1) Ler só o header para descobrir nomes de colunas
                header_reader = pa_csv.open_csv(
                    csv_file,
                    parse_options=parse_options,
                    read_options=read_options,
                )
                col_names = header_reader.schema.names

                # 2) Forçar TODAS as colunas como utf8 (string)
                all_string_types = {name: pa.utf8() for name in col_names}
                convert_opts = pa_csv.ConvertOptions(
                    strings_can_be_null=True,
                    column_types=all_string_types,
                )

                # 3) Reabrir com tipos forçados
                reader = pa_csv.open_csv(
                    csv_file,
                    parse_options=parse_options,
                    read_options=read_options,
                    convert_options=convert_opts,
                )

                # Construir schema de saída: colunas originais (snake_case) + linhagem
                original_schema = reader.schema
                new_names = [
                    name.lower().replace(' ', '_').replace('-', '_').replace('.', '_').replace('(', '').replace(')', '')
                    for name in original_schema.names
                ]
                renamed_schema = pa.schema([
                    pa.field(new_name, original_schema.field(i).type)
                    for i, new_name in enumerate(new_names)
                ])
                # Adicionar colunas de linhagem
                output_schema = renamed_schema
                for linhagem_field in [
                    pa.field('ramo', pa.utf8()),
                    pa.field('grupo_arquivo', pa.utf8()),
                    pa.field('ano', pa.int64()),
                    pa.field('semestre', pa.utf8()),
                    pa.field('arquivo_origem', pa.utf8()),
                ]:
                    output_schema = output_schema.append(linhagem_field)

                writer = pq.ParquetWriter(parquet_file, output_schema, compression='snappy')

                total_rows = 0
                for batch in reader:
                    n = batch.num_rows
                    total_rows += n

                    # Renomear colunas
                    renamed_batch = pa.record_batch(
                        [batch.column(i) for i in range(batch.num_columns)],
                        schema=renamed_schema,
                    )

                    # Adicionar colunas de linhagem
                    renamed_batch = renamed_batch.append_column(
                        'ramo', pa.array([ramo] * n, type=pa.utf8()))
                    renamed_batch = renamed_batch.append_column(
                        'grupo_arquivo', pa.array([grupo] * n, type=pa.utf8()))
                    renamed_batch = renamed_batch.append_column(
                        'ano', pa.array([ano_val] * n, type=pa.int64()))
                    renamed_batch = renamed_batch.append_column(
                        'semestre', pa.array([semestre_val] * n, type=pa.utf8()))
                    renamed_batch = renamed_batch.append_column(
                        'arquivo_origem', pa.array([csv_file.name] * n, type=pa.utf8()))

                    writer.write_batch(renamed_batch)

                writer.close()

                num_cols = len(output_schema)
                size_mb = parquet_file.stat().st_size / 1e6
                sucessos += 1
                print(f"  ✓ {ramo:6} {grupo:8} {periodo:6} → {parquet_file.name:50} "
                      f"({total_rows:>12,} linhas, {num_cols:>3} cols, {size_mb:>8.1f} MB)")

            except Exception as e:
                erros += 1
                print(f"  ✗ Erro ao processar {csv_file.name}: {str(e)[:80]}")

    print(f"\n✓ Padronização completa!")
    print(f"  Sucessos: {sucessos} | Erros: {erros}")


# Executar Etapa D
etapa_d_padronizacao()


ETAPA D - PADRONIZAÇÃO DE ESQUEMA
  ✓ auto   R_AUTO   2020A  → R_AUTO_2020A_2020A_standardized.parquet            (  35,628,198 linhas,  52 cols,   2394.8 MB)
  ✓ auto   R_AUTO   2020B  → R_AUTO_2020B_2020B_standardized.parquet            (  35,854,022 linhas,  52 cols,   2233.3 MB)
  ✓ auto   S_AUTO   2020A  → S_AUTO_2020A_2020A_standardized.parquet            (   3,139,975 linhas,  30 cols,    105.7 MB)
  ✓ auto   S_AUTO   2020B  → S_AUTO_2020B_2020B_standardized.parquet            (   2,454,613 linhas,  30 cols,     84.3 MB)
  ✓ comp   R_COMP   2022   → R_COMP_2022_2022_standardized.parquet              ( 144,458,633 linhas,  21 cols,   2080.3 MB)
  ✓ comp   S_COMP   2022   → S_COMP_2022_2022_standardized.parquet              (     911,357 linhas,  17 cols,     14.1 MB)
  ✓ rural  R_RURAL  2021   → R_RURAL_2021_2021_standardized.parquet             (  12,773,353 linhas,  29 cols,    404.6 MB)
  ✓ rural  S_RURAL  2021   → S_RURAL_2021_2021_standardized.parquet             (     158,

## Seção 5.1: Validação Visual dos Parquets Padronizados

Exibe schema (nome + tipo Arrow) e as primeiras 5 linhas de cada Parquet gerado, para confirmar que:
- O delimitador `;` foi aplicado corretamente (cada campo do CSV virou uma coluna separada)
- Os nomes estão em snake_case
- Todas as colunas de linhagem estão presentes (`ramo`, `grupo_arquivo`, `ano`, `semestre`, `arquivo_origem`)
- **Decisão de tipo**: todas as colunas originais do CSV são `utf8` (string), porque os dados da SUSEP contêm códigos alfanuméricos (`014088-0`), zeros à esquerda significativos (`000001`, `06710000`) e valores sentinela (`.`) em campos aparentemente numéricos

In [9]:
print("="*70)
print("VALIDAÇÃO VISUAL - PARQUETS PADRONIZADOS")
print("="*70)

colunas_linhagem = {'ramo', 'grupo_arquivo', 'ano', 'semestre', 'arquivo_origem'}

for base_folder, base_info in RAMOS.items():
    standardized_path = DATA_ROOT / base_folder / "standardized"
    if not standardized_path.exists():
        continue

    for pq_file in sorted(standardized_path.glob("*.parquet")):
        schema = pq.read_schema(pq_file)
        meta = pq.read_metadata(pq_file)

        print(f"\n{'─'*70}")
        print(f"  {pq_file.name}")
        print(f"   Linhas: {meta.num_rows:,}  |  Colunas: {meta.num_columns}")
        print(f"   Tamanho: {pq_file.stat().st_size / 1e6:.1f} MB")
        print(f"{'─'*70}")

        # Schema: nome + tipo
        print(f"\n  {'#':>3}  {'Coluna':<25} {'Tipo Arrow':<15} {'Obs'}")
        print(f"  {'---':>3}  {'-'*25} {'-'*15} {'-'*12}")
        for i, field in enumerate(schema):
            tag = "<- linhagem" if field.name in colunas_linhagem else ""
            print(f"  {i:>3}  {field.name:<25} {str(field.type):<15} {tag}")

        # Primeiras 5 linhas (ler apenas as primeiras linhas do Parquet)
        tbl = pq.ParquetFile(pq_file).read_row_group(0, columns=schema.names).slice(0, 5)
        print(f"\n  Primeiras 5 linhas:")

        # Calcular larguras de coluna para exibição compacta
        col_widths = {}
        for name in schema.names:
            arr = tbl.column(name)
            max_val_len = max(
                (len(str(arr[i].as_py())) for i in range(arr.length())),
                default=0
            )
            col_widths[name] = max(len(name), min(max_val_len, 18))

        # Header
        print("  " + " | ".join(f"{name:<{col_widths[name]}}" for name in schema.names))
        print("  " + "-+-".join("-" * col_widths[name] for name in schema.names))

        # Rows
        for row_idx in range(tbl.num_rows):
            vals = []
            for name in schema.names:
                v = str(tbl.column(name)[row_idx].as_py())
                w = col_widths[name]
                if len(v) > w:
                    v = v[:w-1] + "~"
                vals.append(f"{v:<{w}}")
            print("  " + " | ".join(vals))

        # Verificar colunas de linhagem
        presentes = colunas_linhagem & set(schema.names)
        faltando = colunas_linhagem - presentes
        if faltando:
            print(f"\n  ! Colunas de linhagem FALTANDO: {', '.join(sorted(faltando))}")
        else:
            print(f"\n  OK - Todas as 5 colunas de linhagem presentes")

print(f"\n{'='*70}")
print("VALIDACAO VISUAL CONCLUIDA")
print("="*70)

VALIDAÇÃO VISUAL - PARQUETS PADRONIZADOS

──────────────────────────────────────────────────────────────────────
  R_AUTO_2020A_2020A_standardized.parquet
   Linhas: 35,628,198  |  Colunas: 52
   Tamanho: 2394.8 MB
──────────────────────────────────────────────────────────────────────

    #  Coluna                    Tipo Arrow      Obs
  ---  ------------------------- --------------- ------------
    0  cod_apo                   string          
    1  cod_endosso               string          
    2  data_comp                 string          
    3  cod_end                   string          
    4  item                      string          
    5  tipo_pes                  string          
    6  modalidade                string          
    7  tipo_prod                 string          
    8  cobertura                 string          
    9  cod_modelo                string          
   10  ano_modelo                string          
   11  cod_tarif                 string         

## Seção 6: Resumo e Validação

In [6]:
print("\n" + "="*70)
print("RESUMO FINAL DO PIPELINE")
print("="*70)

# Validar manifesto
manifest_file = UNIFIED_ROOT / "manifestos" / "links_checksums_manifest.csv"
if manifest_file.exists():
    df_manifest = pd.read_csv(manifest_file)
    print(f"\n✓ Manifesto gerado:")
    print(f"  Arquivo: {manifest_file}")
    print(f"  Registros: {len(df_manifest)}")
    print(f"\n  Distribuição por ramo:")
    print(df_manifest['ramo'].value_counts().to_string())

# Contar arquivos extraídos
print(f"\n✓ Arquivos extraídos:")
for base_folder, base_info in RAMOS.items():
    base_path = DATA_ROOT / base_folder
    extracted_path = base_path / "extracted"
    if extracted_path.exists():
        num_files = len(list(extracted_path.rglob('*.csv')))
        print(f"  {base_info['ramo']:10}: {num_files:>3} arquivos")

# Contar arquivos padronizados
print(f"\n✓ Arquivos padronizados (Parquet):")
parquet_count = 0
for base_folder, base_info in RAMOS.items():
    base_path = DATA_ROOT / base_folder
    standardized_path = base_path / "standardized"
    if standardized_path.exists():
        num_parquet = len(list(standardized_path.glob('*.parquet')))
        parquet_count += num_parquet
        print(f"  {base_info['ramo']:10}: {num_parquet:>3} arquivos")

print(f"\n  Total de Parquets: {parquet_count}")

print("\n" + "="*70)
print("✓ PIPELINE ETAPAS A-D CONCLUÍDO COM SUCESSO!")
print("="*70)


RESUMO FINAL DO PIPELINE

✓ Manifesto gerado:
  Arquivo: data/unified/manifestos/links_checksums_manifest.csv
  Registros: 8

  Distribuição por ramo:
ramo
auto     4
comp     2
rural    2

✓ Arquivos extraídos:
  auto      :   4 arquivos
  comp      :   2 arquivos
  rural     :   2 arquivos

✓ Arquivos padronizados (Parquet):
  auto      :   4 arquivos
  comp      :   2 arquivos
  rural     :   2 arquivos

  Total de Parquets: 8

✓ PIPELINE ETAPAS A-D CONCLUÍDO COM SUCESSO!


## Próximos Passos

As etapas A até D foram concluídas com sucesso! 

**Arquivos gerados:**
- `data/unified/manifestos/links_checksums_manifest.csv` - Catálogo consolidado com checksums
- `data/gov_br/susep/central-de-conteudos/dados-estatisticos/bases-anonimizadas/{bases_*}/extracted/` - Arquivos descompactados
- `data/gov_br/susep/central-de-conteudos/dados-estatisticos/bases-anonimizadas/{bases_*}/standardized/` - Dados padronizados em Parquet

**Etapas restantes (E e posteriores):**
- Etapa E: Merge em Estrutura Única (concatenar datasets padronizados)
- Etapa F: Harmonização de Variáveis (criar dicionário de mapeamento)
- Etapa G: Controle de Qualidade (validar integridade dos dados)
- Etapa H: Exportação Final (Parquet + CSV + Relatório)